---
title: "3 - Query Decomposition for Multi-Year Tesla RAG"
jupyter: python3
execute:
  eval: false
  echo: true
  warning: false
  message: false
---

# Query Decomposition for Multi-Year Tesla RAG

Complex SEC questions often mix several tasks: find evidence, compare years, separate risk categories, and synthesize a business interpretation. Query decomposition turns one broad question into smaller retrieval queries before final answer generation.

This notebook uses the same Tesla 2015-2025 10-K corpus as the baseline notebook.

## Setup

In [1]:
%pip install -q beautifulsoup4 requests pandas faiss-cpu sentence-transformers langchain langchain-community langchain-text-splitters langchain-huggingface langchain-openai langsmith

Note: you may need to restart the kernel to use updated packages.


D:\Repositories\AD698-generative-ai-for-BA\.venv\Scripts\python.exe: No module named pip


## Inline SEC 10-K Helper Code

The next cell is intentionally kept inside the notebook instead of being imported from a separate module. It shows the full mechanics of the RAG data layer:

- the multi-year Tesla 10-K URL list
- SEC download and local caching
- HTML-to-text cleanup
- LangChain `Document` creation with year metadata
- recursive chunking with stable `chunk_id` values
- retrieval diagnostics for keyword coverage and year coverage
- context formatting for grounded answer generation

Keeping this code visible makes the notebooks more demonstrative: the retrieval and evaluation results can be traced back to the exact preprocessing choices.

In [2]:
from pathlib import Path
import os
import shutil
import sys
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

if (Path.cwd() / "help-code" / "Rag").exists():
    RAG_DIR = Path.cwd() / "help-code" / "Rag"
else:
    RAG_DIR = Path.cwd()

IN_COLAB = "google.colab" in sys.modules

# Colab Drive is optional. Uncomment these lines in Colab when you want
# SEC filings and output CSVs mirrored to Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")

DRIVE_RAG_DIR = Path("/content/drive/MyDrive/Colab Notebooks/RAG Models")


def get_secret(name: str, aliases: list[str] | None = None) -> str | None:
    """Return a secret from environment variables or Colab userdata."""
    names = [name, *(aliases or [])]
    for candidate in names:
        value = os.environ.get(candidate)
        if value:
            return value
    try:
        from google.colab import userdata
        for candidate in names:
            value = userdata.get(candidate)
            if value:
                return value
    except Exception:
        pass
    return None


def load_secret_to_env(env_name: str, aliases: list[str] | None = None) -> None:
    value = get_secret(env_name, aliases=aliases)
    if value and env_name not in os.environ:
        os.environ[env_name] = value


load_secret_to_env("OPENAI_API_KEY", aliases=["OPENAI_KEY"])
load_secret_to_env("COHERE_API_KEY", aliases=["COHERE_KEY"])
load_secret_to_env("HF_TOKEN", aliases=["HUGGINGFACE_API_KEY", "HF_KEY"])


def default_sec_cache_dir() -> Path:
    """Choose a cache path that works in Colab and locally."""
    if DRIVE_RAG_DIR.exists():
        return DRIVE_RAG_DIR / "sec-cache" / "sec-edgar-filings"
    repo_root = find_repo_root()
    if (repo_root / "help-code").exists():
        return repo_root / "help-code" / "secfile" / "sec-edgar-filings"
    return Path.cwd() / "sec-cache" / "sec-edgar-filings"


def get_output_dirs() -> tuple[Path, Path | None]:
    local_output_dir = RAG_DIR / "outputs"
    drive_output_dir = DRIVE_RAG_DIR / "outputs" if DRIVE_RAG_DIR.exists() else None
    local_output_dir.mkdir(parents=True, exist_ok=True)
    if drive_output_dir:
        drive_output_dir.mkdir(parents=True, exist_ok=True)
    return local_output_dir, drive_output_dir


def mirror_outputs_to_drive(local_output_dir: Path) -> None:
    _, drive_output_dir = get_output_dirs()
    if not drive_output_dir:
        return
    for path in local_output_dir.glob("*.csv"):
        shutil.copy2(path, drive_output_dir / path.name)

from dataclasses import dataclass
from typing import Iterable
from urllib.parse import urlparse
import re
import time
import numpy as np
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

SEC_USER_AGENT = "AD698-RAG-course/1.0 contact@example.com"

TESLA_10K_URLS = [
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459016013195/tsla-10k_20151231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459017003118/tsla-10k_20161231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459018002956/tsla-10k_20171231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459019003165/tsla-10k_20181231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459020004475/tsla-10k_20191231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459021004599/tsla-10k_20201231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000095017022000796/tsla-20211231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000095017023001409/tsla-20221231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000162828024002390/tsla-20231231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000162828025003063/tsla-20241231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000162828026003952/tsla-20251231.htm",
]

TESLA_EVAL_QUESTIONS = [
    {
        "qid": "q1_battery_technology",
        "user_input": "How did Tesla describe battery technology, battery costs, or battery supply across the filings?",
        "reference": "A strong answer compares multiple years and cites evidence about battery technology, battery cost, supply constraints, production scale, and energy storage or vehicle battery strategy.",
        "reference_keywords": ["battery", "cost", "supply", "technology", "production"],
    },
    {
        "qid": "q2_manufacturing_capacity",
        "user_input": "How did Tesla's manufacturing capacity and production ramp risks evolve from 2015 through 2025?",
        "reference": "A strong answer discusses production ramp, manufacturing capacity, factories, scaling risk, delivery volume, and operational execution across several filing years.",
        "reference_keywords": ["manufacturing", "production", "capacity", "factory", "ramp"],
    },
    {
        "qid": "q3_autopilot_self_driving",
        "user_input": "What do the filings say about Autopilot, self-driving, artificial intelligence, or autonomous vehicle technology?",
        "reference": "A strong answer cites filing evidence about Autopilot, Full Self-Driving, autonomous driving, AI systems, safety, regulatory risk, and product development.",
        "reference_keywords": ["autopilot", "self-driving", "autonomous", "artificial intelligence", "regulatory"],
    },
    {
        "qid": "q4_regulatory_risk",
        "user_input": "What regulatory risks appear repeatedly in Tesla's multi-year 10-K filings?",
        "reference": "A strong answer identifies recurring regulatory themes such as vehicle safety, emissions, energy, consumer protection, data, labor, international operations, and securities or compliance risk.",
        "reference_keywords": ["regulatory", "compliance", "safety", "emissions", "international"],
    },
    {
        "qid": "q5_revenue_business_model",
        "user_input": "How did Tesla's business model and revenue sources change across the filing years?",
        "reference": "A strong answer compares automotive revenue with energy generation and storage, services, leasing, regulatory credits, growth in deliveries, and changes in operating scale.",
        "reference_keywords": ["revenue", "automotive", "energy", "services", "leasing"],
    },
    {
        "qid": "q6_competition",
        "user_input": "How does Tesla describe competitive pressure in electric vehicles, energy storage, and related markets?",
        "reference": "A strong answer cites competition from incumbent automakers, EV entrants, battery suppliers, energy storage providers, technology companies, and pricing or innovation pressure.",
        "reference_keywords": ["competition", "electric vehicles", "energy storage", "pricing", "innovation"],
    },
]

@dataclass(frozen=True)
class FilingRecord:
    ticker: str
    year: int
    url: str
    local_path: Path


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in [start, *start.parents]:
        if (candidate / "help-code").exists() and (candidate / "M5").exists():
            return candidate
    return Path.cwd()


def filing_year_from_url(url: str) -> int:
    match = re.search(r"20\d{2}|19\d{2}", url)
    if not match:
        raise ValueError(f"Could not infer year from URL: {url}")
    return int(match.group(0))


def cache_path_for_url(url: str, cache_dir: Path) -> Path:
    parsed = urlparse(url)
    filename = Path(parsed.path).name
    year = filing_year_from_url(url)
    return cache_dir / "TSLA" / "10-K" / str(year) / filename


def tesla_filing_records(cache_dir: Path | None = None) -> list[FilingRecord]:
    root = find_repo_root()
    cache_dir = default_sec_cache_dir() if cache_dir is None else cache_dir
    return [
        FilingRecord(
            ticker="TSLA",
            year=filing_year_from_url(url),
            url=url,
            local_path=cache_path_for_url(url, cache_dir),
        )
        for url in TESLA_10K_URLS
    ]


def download_filing(record: FilingRecord, user_agent: str = SEC_USER_AGENT, sleep_seconds: float = 0.2) -> Path:
    record.local_path.parent.mkdir(parents=True, exist_ok=True)
    if record.local_path.exists() and record.local_path.stat().st_size > 0:
        return record.local_path
    response = requests.get(record.url, headers={"User-Agent": user_agent}, timeout=60)
    response.raise_for_status()
    record.local_path.write_text(response.text, encoding="utf-8", errors="ignore")
    time.sleep(sleep_seconds)
    return record.local_path


def ensure_tesla_filings(download: bool = True, user_agent: str = SEC_USER_AGENT) -> list[FilingRecord]:
    records = tesla_filing_records()
    if download:
        for record in records:
            download_filing(record, user_agent=user_agent)
    return records


def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "ix:header", "ix:hidden"]):
        tag.decompose()
    text = soup.get_text(" ")
    return re.sub(r"\s+", " ", text).strip()


def load_filing_documents(records: Iterable[FilingRecord], max_chars_per_filing: int | None = None) -> list[Document]:
    docs = []
    for record in records:
        if not record.local_path.exists():
            raise FileNotFoundError(f"Missing cached filing for {record.year}: {record.local_path}")
        html = record.local_path.read_text(encoding="utf-8", errors="ignore")
        text = html_to_text(html)
        if max_chars_per_filing:
            text = text[:max_chars_per_filing]
        docs.append(
            Document(
                page_content=text,
                metadata={
                    "ticker": record.ticker,
                    "filing_year": record.year,
                    "filing_type": "10-K",
                    "source_url": record.url,
                    "source_file": str(record.local_path),
                },
            )
        )
    return docs


def chunk_filings(docs: list[Document], chunk_size: int = 1200, chunk_overlap: int = 180) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["Item 1A.", "Item 7.", "Item 8.", "\n\n", ". ", " "],
    )
    chunks = splitter.split_documents(docs)
    for i, doc in enumerate(chunks):
        year = doc.metadata.get("filing_year", "unknown")
        ticker = doc.metadata.get("ticker", "TSLA")
        doc.metadata["chunk_id"] = f"{ticker}-{year}-{i:05d}"
    return chunks


def chunk_audit(chunks: list[Document]) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "chunk_id": doc.metadata.get("chunk_id"),
            "ticker": doc.metadata.get("ticker"),
            "filing_year": doc.metadata.get("filing_year"),
            "n_chars": len(doc.page_content),
            "preview": doc.page_content[:180],
        }
        for doc in chunks
    )


def format_context(docs: list[Document], max_chars_per_doc: int = 1600) -> str:
    return "\n\n".join(
        f"[{doc.metadata.get('ticker')} {doc.metadata.get('filing_year')} {doc.metadata.get('chunk_id')}]\n"
        f"{doc.page_content[:max_chars_per_doc]}"
        for doc in docs
    )


def keyword_hit(text: str, keyword: str) -> bool:
    return keyword.lower() in re.sub(r"\s+", " ", text.lower())


def retrieval_diagnostics(retrieved_docs: list[Document], reference_keywords: list[str]) -> dict:
    combined = " ".join(doc.page_content for doc in retrieved_docs)
    keyword_hits = {kw: keyword_hit(combined, kw) for kw in reference_keywords}
    rank_weighted_hits = []
    for rank, doc in enumerate(retrieved_docs, start=1):
        hits = sum(keyword_hit(doc.page_content, kw) for kw in reference_keywords)
        rank_weighted_hits.append(hits / rank)
    years = [doc.metadata.get("filing_year") for doc in retrieved_docs]
    return {
        "retrieved_years": years,
        "unique_years_retrieved": len(set(years)),
        "keyword_coverage": float(np.mean(list(keyword_hits.values()))) if keyword_hits else np.nan,
        "rank_weighted_keyword_score": float(np.sum(rank_weighted_hits)),
        "keyword_hits": keyword_hits,
    }


def extractive_fallback_answer(question: str, retrieved_docs: list[Document], max_chars_per_doc: int = 500) -> str:
    parts = [f"Question: {question}", "Retrieved Tesla filing evidence:"]
    for doc in retrieved_docs[:4]:
        parts.append(
            f"[TSLA {doc.metadata.get('filing_year')} {doc.metadata.get('chunk_id')}] "
            f"{doc.page_content[:max_chars_per_doc]}"
        )
    return "\n\n".join(parts)

C:\Users\nakulpadalkar\AppData\Local\Temp\ipykernel_45232\1121230134.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Optional LangSmith Tracing

In [3]:
if "LANGSMITH_API_KEY" in os.environ:
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "ad698-tesla-query-decomposition"
else:
    print("LANGSMITH_API_KEY is not set. The notebook still runs without tracing.")

LANGSMITH_API_KEY is not set. The notebook still runs without tracing.


## Build the Shared Retriever

In [4]:
records = ensure_tesla_filings(download=True)
docs = load_filing_documents(records)
chunks = chunk_filings(docs, chunk_size=1200, chunk_overlap=180)

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = FAISS.from_documents(chunks, embedding_model)
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 6, "fetch_k": 30})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Decompose Questions

In [5]:
decomposition_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        (
            "Break the user's SEC filing question into 3 to 5 focused retrieval queries. "
            "Return one query per line. Include year-comparison queries when useful."
        ),
    ),
    ("user", "{question}"),
])

if "OPENAI_API_KEY" in os.environ:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    decomposer = decomposition_prompt | llm | StrOutputParser()
else:
    decomposer = None

def heuristic_decompose(question: str) -> list[str]:
    return [
        question,
        f"Tesla 10-K early years evidence for: {question}",
        f"Tesla 10-K recent years evidence for: {question}",
        f"Tesla risks and business changes related to: {question}",
    ]

def decompose_question(question: str) -> list[str]:
    if decomposer is None:
        return heuristic_decompose(question)
    text = decomposer.invoke({"question": question})
    queries = [line.strip("- 0123456789.").strip() for line in text.splitlines()]
    queries = [q for q in queries if q]
    return queries[:5] if queries else heuristic_decompose(question)

In [6]:
question = TESLA_EVAL_QUESTIONS[1]["user_input"]
subqueries = decompose_question(question)
subqueries

["How did Tesla's manufacturing capacity and production ramp risks evolve from 2015 through 2025?",
 "Tesla 10-K early years evidence for: How did Tesla's manufacturing capacity and production ramp risks evolve from 2015 through 2025?",
 "Tesla 10-K recent years evidence for: How did Tesla's manufacturing capacity and production ramp risks evolve from 2015 through 2025?",
 "Tesla risks and business changes related to: How did Tesla's manufacturing capacity and production ramp risks evolve from 2015 through 2025?"]

## Retrieve for Each Subquery

In [7]:
def retrieve_decomposed(question: str, max_docs: int = 10):
    seen = set()
    selected = []
    trace_rows = []

    for subquery in decompose_question(question):
        docs_for_subquery = retriever.invoke(subquery)
        for rank, doc in enumerate(docs_for_subquery, start=1):
            chunk_id = doc.metadata["chunk_id"]
            trace_rows.append(
                {
                    "subquery": subquery,
                    "rank": rank,
                    "year": doc.metadata["filing_year"],
                    "chunk_id": chunk_id,
                    "preview": doc.page_content[:220],
                }
            )
            if chunk_id not in seen and len(selected) < max_docs:
                selected.append(doc)
                seen.add(chunk_id)

    return selected, pd.DataFrame(trace_rows)

In [8]:
decomposed_docs, trace_df = retrieve_decomposed(question)
trace_df.head(20)

,subquery,rank,year,chunk_id,preview
0,How did Tesla's manufacturing capacity and pro...,1,2024,TSLA-2024-04615,". For example, during the first quarter of 202..."
1,How did Tesla's manufacturing capacity and pro...,2,2015,TSLA-2015-00153,. We expect our annual production will increas...
2,How did Tesla's manufacturing capacity and pro...,3,2000,TSLA-2000-03229,. We may also experience similar future delays...
3,How did Tesla's manufacturing capacity and pro...,4,1900,TSLA-1900-01675,". We broke ground in January 2019, and subject..."
4,How did Tesla's manufacturing capacity and pro...,5,2024,TSLA-2024-04613,". 32 Management Opportunities, Challenges and ..."
5,How did Tesla's manufacturing capacity and pro...,6,2017,TSLA-2017-01107,". With demand outpacing production, we expect ..."
6,Tesla 10-K early years evidence for: How did T...,1,2024,TSLA-2024-04615,". For example, during the first quarter of 202..."
7,Tesla 10-K early years evidence for: How did T...,2,2022,TSLA-2022-03682,". In 2022, we had over 200 graduates from Tesl..."
8,Tesla 10-K early years evidence for: How did T...,3,2015,TSLA-2015-00113,. Our financial results may vary significantly...
9,Tesla 10-K early years evidence for: How did T...,4,2015,TSLA-2015-00150,. We commenced customer deliveries of our Mode...


## Generate a Synthesized Answer

In [9]:
synthesis_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        (
            "Answer using only the retrieved Tesla 10-K context. "
            "When possible, describe how the evidence changes across years. "
            "Cite filing years and chunk IDs."
        ),
    ),
    ("user", "Question:\n{question}\n\nContext:\n{context}"),
])

if "OPENAI_API_KEY" in os.environ:
    answer_chain = synthesis_prompt | llm | StrOutputParser()
    decomposed_answer = answer_chain.invoke(
        {"question": question, "context": format_context(decomposed_docs)}
    )
else:
    decomposed_answer = extractive_fallback_answer(question, decomposed_docs)

print(decomposed_answer)

Question: How did Tesla's manufacturing capacity and production ramp risks evolve from 2015 through 2025?

Retrieved Tesla filing evidence:

[TSLA 2024 TSLA-2024-04615] . For example, during the first quarter of 2024, we experienced a sequential decline in production volumes partially caused by the early phase of the production ramp of the updated Model 3 at our Fremont factory, and factory shutdowns at Gigafactory Berlin-Brandenburg resulting from shipping diversions caused by the Red Sea conflict and an arson attack. In the first quarter of 2025, as we launch our New Model Y worldwide, we may similarly experience delays or declines in production volumes due t

[TSLA 2015 TSLA-2015-00153] . We expect our annual production will increase considerably each year for the next several years. In addition, scaling our deliveries means that we will have an increasing number of cars in-transit, therefore production will exceed deliveries. During 2015, we achieved significant efficiencies in Mod

## Compare Baseline vs Decomposed Retrieval

In [10]:
comparison_rows = []

for item in TESLA_EVAL_QUESTIONS:
    baseline_docs = retriever.invoke(item["user_input"])
    decomposed_docs, _ = retrieve_decomposed(item["user_input"])

    for method, docs_for_q in {
        "single_query_mmr": baseline_docs,
        "decomposed_query_mmr": decomposed_docs,
    }.items():
        diag = retrieval_diagnostics(docs_for_q, item["reference_keywords"])
        comparison_rows.append(
            {
                "qid": item["qid"],
                "method": method,
                "retrieved_years": diag["retrieved_years"],
                "unique_years_retrieved": diag["unique_years_retrieved"],
                "keyword_coverage": diag["keyword_coverage"],
                "rank_weighted_keyword_score": diag["rank_weighted_keyword_score"],
                "chunk_ids": [doc.metadata["chunk_id"] for doc in docs_for_q],
            }
        )

decomposition_compare_df = pd.DataFrame(comparison_rows)
decomposition_compare_df

,qid,method,retrieved_years,unique_years_retrieved,keyword_coverage,rank_weighted_keyword_score,chunk_ids
0,q1_battery_technology,single_query_mmr,"[2016, 2000, 1900, 1900, 2017, 2015]",5,0.8,3.916667,"[TSLA-2016-00489, TSLA-2000-03314, TSLA-1900-0..."
1,q1_battery_technology,decomposed_query_mmr,"[2016, 2000, 1900, 1900, 2017, 2015, 2000, 202...",6,0.8,4.811111,"[TSLA-2016-00489, TSLA-2000-03314, TSLA-1900-0..."
2,q2_manufacturing_capacity,single_query_mmr,"[2024, 2015, 2000, 1900, 2024, 2017]",5,1.0,8.633333,"[TSLA-2024-04615, TSLA-2015-00153, TSLA-2000-0..."
3,q2_manufacturing_capacity,decomposed_query_mmr,"[2024, 2015, 2000, 1900, 2024, 2017, 2022, 201...",7,1.0,10.177381,"[TSLA-2024-04615, TSLA-2015-00153, TSLA-2000-0..."
4,q3_autopilot_self_driving,single_query_mmr,"[2024, 2025, 2025, 2025, 1900, 2017]",4,0.8,4.483333,"[TSLA-2024-04582, TSLA-2025-05225, TSLA-2025-0..."
5,q3_autopilot_self_driving,decomposed_query_mmr,"[2024, 2025, 2025, 2025, 1900, 2017, 1900, 202...",5,1.0,5.111905,"[TSLA-2024-04582, TSLA-2025-05225, TSLA-2025-0..."
6,q4_regulatory_risk,single_query_mmr,"[2000, 2022, 2023, 2023, 2025, 2015]",5,0.2,0.250000,"[TSLA-2000-03223, TSLA-2022-03710, TSLA-2023-0..."
7,q4_regulatory_risk,decomposed_query_mmr,"[2000, 2022, 2023, 2023, 2025, 2015, 2024, 190...",7,0.4,0.350000,"[TSLA-2000-03223, TSLA-2022-03710, TSLA-2023-0..."
8,q5_revenue_business_model,single_query_mmr,"[2015, 2024, 2025, 2015, 2023, 1900]",5,1.0,7.583333,"[TSLA-2015-00181, TSLA-2024-04577, TSLA-2025-0..."
9,q5_revenue_business_model,decomposed_query_mmr,"[2015, 2024, 2025, 2015, 2023, 1900, 2016, 200...",7,1.0,8.370635,"[TSLA-2015-00181, TSLA-2024-04577, TSLA-2025-0..."


In [11]:
decomposition_compare_df.groupby("method").agg(
    avg_keyword_coverage=("keyword_coverage", "mean"),
    avg_unique_years=("unique_years_retrieved", "mean"),
    avg_rank_weighted_keyword_score=("rank_weighted_keyword_score", "mean"),
)

,avg_keyword_coverage,avg_unique_years,avg_rank_weighted_keyword_score
method,,,
decomposed_query_mmr,0.833333,6.333333,5.637500
single_query_mmr,0.766667,5.000000,4.877778


## Save Artifacts

In [12]:
output_dir, drive_output_dir = get_output_dirs()

decomposition_compare_df.to_csv(output_dir / "tesla_query_decomposition_compare.csv", index=False)
trace_df.to_csv(output_dir / "tesla_query_decomposition_trace_example.csv", index=False)

mirror_outputs_to_drive(output_dir)

output_dir, drive_output_dir

(WindowsPath('D:/Repositories/AD698-generative-ai-for-BA/help-code/Rag/outputs'),
 None)